# Liu2021 GSE131354 内部最高候选的探索性评分

按用户决定，在本组四个候选均未通过原内部验收时，使用内部 miLISI 最高的既有 checkpoint。生成沿用原训练缺失掩码与联合秩映射；正式验收记录保留，结果表标注探索性。

In [ ]:
from pathlib import Path
import json
import os
import anndata as ad
import numpy as np
import pandas as pd
import torch
from data.paths import OUTPUTS, check_output, require_environment
from data.training import load_target_labels
from model.src.generation import write_synthetic_h5ad, validate_synthetic_h5ad
from model.src.model import load_cvae_model
from model.src.pipeline import CASES, write_status
from model.src.postprocessing import copy_checkpoint, finalize_model, write_json, write_training_mask_h5ad
from model.src.training import selection_rank

require_environment()
case = 'liu2021/GSE131354'
root = check_output(OUTPUTS/'min10')
dataset = root/'model'/case
prepared = root/'preprocessing'/case
records = {
    name: json.loads((dataset/name/'diagnostics.json').read_text(encoding='utf-8'))
    for name, _, _ in CASES
}
assert not any(record['selection_eligible'] for record in records.values())
name = min(records,key=lambda name: selection_rank(records[name]))
record = records[name]
assert name == 'frozen_beta_1e6'
configuration = json.loads((dataset/'config.json').read_text(encoding='utf-8'))['配置']
device = torch.device(os.environ.get('EXPLORATORY_DEVICE','cuda:2'))
torch.set_num_threads(8)
training = ad.read_h5ad(prepared/'TrainingReal.h5ad',backed='r')
matrix = np.load(dataset/'work/values.float32.npy',mmap_mode='r')
try:
    audit = pd.read_csv(dataset/'work/input_cell_audit.csv')
    labels = training.obs['Subclass'].astype(str).to_numpy()
    assert training.shape == matrix.shape
    np.testing.assert_array_equal(training.obs_names.to_numpy(),audit['cell_id'].to_numpy())
    np.testing.assert_array_equal(labels,audit['Subclass'].to_numpy())
    with np.load(dataset/'internal_reference.npz') as reference:
        fit = reference['fit_rows'].copy()
        validation = reference['validation_rows'].copy()
    names = training.var_names.to_numpy()
    model, checkpoint = load_cvae_model(Path(record['model_path']),device)
    target = load_target_labels(prepared/'HeldoutReal.h5ad',names,checkpoint['class_names'])
    seed = configuration['SEED']
    np.random.default_rng(seed).shuffle(target)
    indices = np.searchsorted(checkpoint['class_names'],target)
    raw_path = dataset/'exploratory_raw/data.h5ad'
    if not raw_path.is_file():
        write_synthetic_h5ad(raw_path,model,indices,target,training.var,names,checkpoint['sampling_steps'],
                             device,seed,configuration['GENERATION_ROW_BLOCK'],configuration['FEATURE_BLOCK'])
    validate_synthetic_h5ad(raw_path,names,len(target))
    del model
    torch.cuda.empty_cache()
    masked_path = dataset/'CVAE_training_mask/data.h5ad'
    if not masked_path.is_file():
        write_training_mask_h5ad(raw_path,masked_path,matrix,labels,fit,seed)
    checkpoint_path = masked_path.with_name('model.pt')
    if not checkpoint_path.is_file():
        copy_checkpoint(record['model_path'],checkpoint_path)
    result = finalize_model(dataset,matrix,labels,fit,validation,names,masked_path,checkpoint_path)
finally:
    training.file.close()
    matrix._mmap.close()

report = {
    '状态':'探索性生成完成','结果口径':'探索性','入选候选':name,'通过原验收':False,
    '选模依据':'仅按内部验证 miLISI 排名','内部验证miLISI':record['selection_milisi'],
    '内部验证近邻数':record['selection_neighbors'],'数据':str(result),
}
write_json(dataset/'exploratory.json',report)
write_status(dataset,'探索性生成完成',**report)
job = root/'internal/jobs/liu2021__GSE131354__simscDNAm.json'
payload = {'状态':'完成','数据集':case,'方法':'simscDNAm','退出码':0,'生成文件':str(result),'结果口径':'探索性'}
temporary=job.with_suffix('.json.partial')
temporary.write_text(json.dumps(payload,ensure_ascii=False),encoding='utf-8')
temporary.replace(job)
print(report,flush=True)